In [ ]:
!nvidia-smi

Thu Mar 26 05:35:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             27W /   70W |    1151MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Set restart = True to train from scratch
# Set restart = False to continue training from a checkpoint
restart = False
epoch_to_pickup = 0
author_to_pickup = 'poe'  # 'austen', 'poe', 'twain'

# Authors to train (in order)
authors = ['austen', 'poe', 'twain']

In [ ]:
# Import libraries
from tensorflow.keras.layers import StringLookup
import numpy as np
import os
import random
import contextlib
import io
import re
import string
import gc

import tensorflow as tf
from tensorflow.keras.layers import Dense, Embedding
from tensorflow.keras.layers import TextVectorization

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [ ]:
# Celda 4
from google.colab import drive
drive.mount('/content/drive')

# Base path en Google Drive
path = '/content/drive/My Drive/cse450_publishing/'

# Crear carpetas por autor si no existen
import os
for author in authors:
    os.makedirs(path + author, exist_ok=True)

print("Google Drive montado y carpetas listas ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montado y carpetas listas ✅


## Functions for downloading text

In [ ]:
# def preprocess_text(text):

#     text = text.replace("Project Gutenberg", "")
#     text = text.replace("Gutenberg", "")

#     # Remove carriage returns
#     text = text.replace("\r", "")

#     # fix quotes
#     text = text.replace("“", "\"")
#     text = text.replace("”", "\"")

#     # Replace any capital letter at the start of a word with ^ followed by the lowercase letter
#     text = re.sub(r"(?<![a-zA-Z])([A-Z])", lambda match: f"^{match.group(0).lower()}", text)

#     # Replace all other capital letters with lowercase
#     text = re.sub(r"([A-Z])", lambda match: f"{match.group(0).lower()}", text)

#     # Remove duplicate whitespace
#     text = re.sub(r"\s+", " ", text)
#     text = re.sub(r"\n+", "\n", text)
#     text = re.sub(r"\t+", "\t", text)

#     # Replace whitespace characters with special words
#     text = re.sub(r"(\t)", r" zztabzz ", text)
#     text = re.sub(r"(\n)", r" zznewlinezz ", text)
#     text = re.sub(r"(\s)", r" zzspacezz ", text)

#     # Split before and after punctuation
#     for punctuation in string.punctuation:
#         text = text.replace(punctuation, f" {punctuation} ")

#     return text

In [ ]:
def preprocess_text(text):
    # Remove Project Gutenberg references
    text = text.replace("Project Gutenberg", "")
    text = text.replace("Gutenberg", "")

    # Remove carriage returns
    text = text.replace("\r", "")

    # Fix quotes
    text = text.replace("\u201c", "\"")
    text = text.replace("\u201d", "\"")

    # Replace capital letters with ^ + lowercase
    text = re.sub(r"(?<![a-zA-Z])([A-Z])", lambda match: f"^{match.group(0).lower()}", text)
    text = re.sub(r"([A-Z])", lambda match: f"{match.group(0).lower()}", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()

In [ ]:
def postprocess_text(text):
    # Remake capital letters
    text = re.sub(r"\^([a-z])", lambda match: f"{match.group(1).upper()}", text)
    text = text.replace("^", "")
    return text

In [ ]:
AUTHOR_URLS = {
    'austen': [
        'https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/austen/austen.txt',
        # No extra URLs needed - already enough data
    ],
    'poe': [
        'https://www.gutenberg.org/cache/epub/2147/pg2147.txt',
        'https://www.gutenberg.org/cache/epub/25525/pg25525.txt',
        'https://www.gutenberg.org/cache/epub/1063/pg1063.txt',
        'https://www.gutenberg.org/cache/epub/2148/pg2148.txt',
    ],
    'twain': [
        'https://www.gutenberg.org/cache/epub/74/pg74.txt',
        'https://www.gutenberg.org/cache/epub/76/pg76.txt',
        'https://www.gutenberg.org/cache/epub/1837/pg1837.txt',
        'https://www.gutenberg.org/cache/epub/102/pg102.txt',
    ]
}

AUTHOR_NAMES = {
    'austen': 'Jane Austen',
    'poe': 'Edgar Allan Poe',
    'twain': 'Mark Twain'
}

def extract_gutenberg_text(text):
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

    start = text.find(start_marker)
    end = text.find(end_marker)

    if start != -1 and end != -1:
        return text[start + len(start_marker):end]
    else:
        print("Warning: Gutenberg markers not found, using full text")
        return text

def clean_gutenberg_text(text, author):
    full_name = AUTHOR_NAMES[author]

    # Eliminar atribuciones del autor
    text = re.sub(rf'by {full_name}', '', text, flags=re.IGNORECASE)

    # Eliminar líneas que son solo números (páginas) o solo espacios
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    return text

def get_text(author):
    if author not in AUTHOR_URLS:
        raise ValueError(f"Author '{author}' not recognized. Choose from: {list(AUTHOR_URLS.keys())}")

    all_text = ''
    urls = AUTHOR_URLS[author]

    for i, file_url in enumerate(urls):
        file_name = f'{author}_{i}.txt'
        local_path = os.path.join('saved_files', file_name)

        try:
            if not os.path.exists('saved_files'):
                os.makedirs('saved_files')

            if os.path.exists(local_path):
                print(f"File '{file_name}' found locally. Using it.")
            else:
                print(f"File '{file_name}' not found locally. Downloading it.")
                downloaded_path = tf.keras.utils.get_file(file_name, file_url)
                with open(downloaded_path, 'rb') as source_file:
                    with open(local_path, 'wb') as dest_file:
                        dest_file.write(source_file.read())

            with open(local_path, 'rb') as file:
                text = file.read().decode(encoding='utf-8')

            # Clean Gutenberg text except for the original austen source
            if not (author == 'austen' and i == 0):
                text = extract_gutenberg_text(text)
                text = clean_gutenberg_text(text, author)

            all_text += text
            print(f"✅ {file_name}: {len(text)} characters added")

        except Exception as e:
            print(f"⚠️ Failed to load {file_name}: {e}")
            import traceback; traceback.print_exc()
            continue

    if not all_text:
        print(f"❌ No text loaded for {author}")
        return None

    print(f"✅ Total text for {author}: {len(all_text)} characters")
    return preprocess_text(all_text)

In [ ]:
# Test download for each author
for author in authors:
    print(f"\nTesting download for {author}...")
    text = get_text(author)
    if text:
        print(f"✅ {author}: {len(text)} characters loaded")
    else:
        print(f"❌ {author}: failed to load")
    del text
gc.collect()


Testing download for austen...
File 'austen_0.txt' found locally. Using it.
✅ austen_0.txt: 4906787 characters added
✅ Total text for austen: 4906787 characters
✅ austen: 4976561 characters loaded

Testing download for poe...
File 'poe_0.txt' found locally. Using it.
✅ poe_0.txt: 580717 characters added
File 'poe_1.txt' found locally. Using it.
✅ poe_1.txt: 2868247 characters added
File 'poe_2.txt' found locally. Using it.
✅ poe_2.txt: 13510 characters added
File 'poe_3.txt' found locally. Using it.
✅ poe_3.txt: 612462 characters added
✅ Total text for poe: 4074936 characters
✅ poe: 3680728 characters loaded

Testing download for twain...
File 'twain_0.txt' found locally. Using it.
✅ twain_0.txt: 401669 characters added
File 'twain_1.txt' found locally. Using it.
✅ twain_1.txt: 583054 characters added
File 'twain_2.txt' found locally. Using it.
✅ twain_2.txt: 404167 characters added
File 'twain_3.txt' found locally. Using it.
✅ twain_3.txt: 300717 characters added
✅ Total text for twa

0

In [ ]:
# getRandomText() was used to augment training data with random Gutenberg books
# This approach is no longer used as each model should learn a specific author's style
# Kept here in case generic English language training is needed in the future

# def getRandomText(numbooks = 1, verbose=False):
#   download_log = io.StringIO()
#   text_random = ''
#   for b in range(numbooks):
#     foundbook = False
#     while(foundbook == False):
#       booknum = random.randint(100,60000)
#       if verbose:
#         print('Trying Book #: ',booknum)
#       if random.random() > 0.5:
#         url = 'https://www.gutenberg.org/files/' + str(booknum) + '/' + str(booknum) + '-0.txt'
#         filename_temp = str(booknum) + '-0.txt'
#       else:
#         url = 'https://www.gutenberg.org/cache/epub/' + str(booknum) + '/pg' + str(booknum) + '.txt'
#         filename_temp = 'pg' + str(booknum) + '.txt'
#       ...

In [ ]:
# Build shared vocabulary using all authors
# This is done once before the training loop
if restart:
    print("Building shared vocabulary...")
    all_text = ''
    for author in authors:
        print(f"Loading {author} text...")
        author_text = get_text(author)
        if author_text:
            all_text += author_text
            print(f"✅ {author}: added {len(author_text)} characters")
        del author_text
        gc.collect()

    print(f"\nTotal text for vocabulary: {len(all_text)} characters")

Building shared vocabulary...
Loading austen text...
File 'austen_0.txt' found locally. Using it.
✅ austen_0.txt: 4906787 characters added
✅ Total text for austen: 4906787 characters
✅ austen: added 4976561 characters
Loading poe text...
File 'poe_0.txt' found locally. Using it.
✅ poe_0.txt: 580717 characters added
File 'poe_1.txt' found locally. Using it.
✅ poe_1.txt: 2868247 characters added
File 'poe_2.txt' found locally. Using it.
✅ poe_2.txt: 13510 characters added
File 'poe_3.txt' found locally. Using it.
✅ poe_3.txt: 612462 characters added
✅ Total text for poe: 4074936 characters
✅ poe: added 3680728 characters
Loading twain text...
File 'twain_0.txt' found locally. Using it.
✅ twain_0.txt: 401669 characters added
File 'twain_1.txt' found locally. Using it.
✅ twain_1.txt: 583054 characters added
File 'twain_2.txt' found locally. Using it.
✅ twain_2.txt: 404167 characters added
File 'twain_3.txt' found locally. Using it.
✅ twain_3.txt: 300717 characters added
✅ Total text for tw

## Make Shared Vocabulary (Adapted from TensorFlow word embedding tutorial)

Build a single vocabulary from all authors combined. This ensures that words
from Austen, Poe, and Twain are all represented in the same vocabulary space,
allowing the same model architecture to be used for all three authors.

In [ ]:
# Hyperparameters
# Keeping all hyperparameters here makes it easy to experiment

# Vocabulary
# No vocab_size needed - character vocabulary is built automatically per author
sequence_length = 300  # increased from 128 words to 100 characters

# Model architecture
embedding_dim = 256    # increased because characters need richer representation
rnn_units = 1024       # increased to capture character-level patterns better

# Training
BATCH_SIZE = 64
BUFFER_SIZE = 10000
num_epochs_total = 10  # increased because character models need more epochs
learning_rate = 0.002
learning_rate_decay = 0.99

# Text generation
temperatures = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sample_prompt = 'The world seemed like such a peaceful place until the magic tree was discovered in London.'

In [ ]:
def build_char_vocabulary(text):
    # Get all unique characters in the text
    vocabulary = sorted(set(text))
    print(f"Vocabulary size: {len(vocabulary)} characters")
    print(f"Characters: {''.join(vocabulary)}")

    # Create character to index and index to character mappings
    char_to_idx = {char: idx for idx, char in enumerate(vocabulary)}
    idx_to_char = {idx: char for idx, char in enumerate(vocabulary)}

    return vocabulary, char_to_idx, idx_to_char

In [ ]:
# if restart:
#     # Create the vectorization layer with our shared vocabulary settings
#     # standardize='lower' converts all text to lowercase
#     # split='whitespace' splits on our custom whitespace tokens (zzspacezz, zztabzz, etc.)
#     # max_tokens limits vocabulary to vocab_size most common words
#     vectorize_layer = TextVectorization(
#         standardize='lower',
#         split='whitespace',
#         max_tokens=vocab_size,
#         output_mode='int',
#         )

In [ ]:
# if restart:
#     print("Adapting vocabulary to all authors...")
#     vectorize_layer.adapt([all_text]) # Build the dictionary.
#     print(f"✅ Vocabulary built with {vocab_size} tokens")

#     # Free memory, all_text no longer needed after vocabulary is built
#     del all_text
#     gc.collect()

In [ ]:
# if restart:
#     vocabulary = vectorize_layer.get_vocabulary()
#     print(f"✅ Vocabulary size: {len(vocabulary)} tokens")
#     print(f"First 10 tokens: {vocabulary[:10]}")
#     print(f"Last 10 tokens: {vocabulary[-10:]}")

## Save Shared Vocabulary

Save the shared vocabulary to disk so it can be reloaded when restart = False.
This only needs to be done once since all authors share the same vocabulary.

In [ ]:
# if restart:
#     vocab_path = path + "vocabulary.txt"
#     with open(vocab_path, "w") as file:
#         for word in vocabulary:
#             file.write(word + "\n")
#     print(f"✅ Vocabulary saved to {vocab_path}")

In [ ]:
def save_vocabulary(vocabulary, author):
    vocab_path = path + f"{author}/vocabulary.txt"
    with open(vocab_path, "w", encoding='utf-8') as f:
        for char in vocabulary:
            f.write(char + "\n")
    print(f"✅ Vocabulary saved for {author} at {vocab_path}")

## Load Saved Vocabulary

When restart = False, load the previously saved shared vocabulary from Google Drive
instead of rebuilding it from scratch.

In [ ]:
# if restart == False:
#     vocab_path = path + "vocabulary.txt"

#     if not os.path.exists(vocab_path):
#         raise FileNotFoundError(f"Vocabulary file not found at {vocab_path}. Run with restart=True first.")

#     with open(vocab_path, "r") as file:
#         vocabulary = [word.strip() for word in file.readlines()]

#     vectorize_layer = TextVectorization(
#         vocabulary=vocabulary,
#         standardize='lower',
#         split='whitespace',
#         max_tokens=vocab_size,
#         output_mode='int',
#         )
#     print(f"✅ Vocabulary loaded: {len(vocabulary)} tokens")

In [ ]:
# print(f"First 20 tokens: {vocabulary[:20]}")
# print(f"Last 20 tokens: {vocabulary[-20:]}")
# print(f"Total vocabulary size: {len(vocabulary)}")

In [ ]:
def load_vocabulary(author):
    vocab_path = path + f"{author}/vocabulary.txt"

    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f"Vocabulary not found at {vocab_path}. Run with restart=True first.")

    with open(vocab_path, "r", encoding='utf-8') as f:
        vocabulary = [line.rstrip("\n") for line in f.readlines()]

    char_to_idx = {char: idx for idx, char in enumerate(vocabulary)}
    idx_to_char = {idx: char for idx, char in enumerate(vocabulary)}

    print(f"✅ Vocabulary loaded for {author}: {len(vocabulary)} characters")
    return vocabulary, char_to_idx, idx_to_char

## Turn Text into a Dataset

Convert raw text into input/target sequence pairs for training.
This will be called separately for each author during the training loop.

In [ ]:
def split_input_target(sequence):
    input_ids = sequence[:-1]
    target_ids = sequence[1:]
    return input_ids, target_ids

def text_to_dataset(text, author, char_to_idx):
    print(f"Converting {author} text to dataset...")

    # Convert all characters to indices
    all_ids = [char_to_idx.get(char, 0) for char in text]

    # Convert to tensor
    all_ids = tf.constant(all_ids, dtype=tf.int64)

    # Create dataset from tensor
    ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)
    del all_ids

    # Create sequences of sequence_length + 1
    sequences = ids_dataset.batch(sequence_length + 1, drop_remainder=True)
    del ids_dataset

    # Split into input/target pairs
    dataset = sequences.map(split_input_target)
    del sequences

    print(f"✅ Dataset ready for {author}")
    return dataset

## Test Dataset Creation

Test that the dataset creation works correctly for each author
before starting the full training loop.

In [ ]:
# if restart:
#     for author in authors:
#         print(f"\nTesting dataset creation for {author}...")
#         test_text = get_text(author)
#         if test_text:
#             test_ds = text_to_dataset(test_text, author)
#             # Verify dataset shape
#             for input_example, target_example in test_ds.take(1):
#                 print(f"Input shape: {input_example.shape}")
#                 print(f"Target shape: {target_example.shape}")
#             del test_ds
#             del test_text
#             gc.collect()
#             print(f"✅ Dataset test passed for {author}")
#         else:
#             print(f"❌ Failed to load text for {author}")

In [ ]:
def text_from_ids(ids, idx_to_char):
    return postprocess_text(''.join([idx_to_char.get(idx, '') for idx in ids.numpy()]))

# words_from_ids is no longer needed - replaced by text_from_ids with idx_to_char
# StringLookup is no longer needed for character-level models
print("✅ Text decoder ready")

✅ Text decoder ready


In [ ]:
if restart:
    print("Testing text decoder...")
    test_text = get_text('austen')
    if test_text:
        # Build vocabulary first before creating dataset
        test_vocab, test_char_to_idx, test_idx_to_char = build_char_vocabulary(test_text)
        test_ds = text_to_dataset(test_text, 'austen', test_char_to_idx)

        for input_example, target_example in test_ds.take(1):
            print("Input IDs: ")
            print(input_example)
            print("Input text: ")
            print(text_from_ids(input_example, test_idx_to_char))
            print("\nTarget IDs: ")
            print(target_example)
            print("Target text: ")
            print(text_from_ids(target_example, test_idx_to_char))

        del test_ds
        del test_text
        del test_vocab
        del test_char_to_idx
        del test_idx_to_char
        gc.collect()
        print("✅ Text decoder test passed")

Testing text decoder...
File 'austen_0.txt' found locally. Using it.
✅ austen_0.txt: 4906787 characters added
✅ Total text for austen: 4906787 characters
Vocabulary size: 58 characters
Characters:  !"&'()*,-.0123456789:;?[]^_abcdefghijklmnopqrstuvwxyz£é‘’
Converting austen text to dataset...
✅ Dataset ready for austen
Input IDs: 
tf.Tensor(
[26 49 42 39 48 40 32  0 26 36  0 26 30 35 28 43 47 32 45  0 26 36  0 26
 32 40 40 28  0 26 50 42 42 31 35 42 48 46 32  8  0 35 28 41 31 46 42 40
 32  8  0 30 39 32 49 32 45  8  0 28 41 31  0 45 36 30 35  8  0 50 36 47
 35  0 28  0 30 42 40 33 42 45 47 28 29 39 32  0 35 42 40 32  0 28 41 31
  0 35 28 43 43 52  0 31 36 46 43 42 46 36 47 36 42 41  8  0 46 32 32 40
 32 31  0 47 42  0 48 41 36 47 32  0 46 42 40 32  0 42 33  0 47 35 32  0
 29 32 46 47  0 29 39 32 46 46 36 41 34 46  0 42 33  0 32 51 36 46 47 32
 41 30 32 22  0 28 41 31  0 35 28 31  0 39 36 49 32 31  0 41 32 28 45 39
 52  0 47 50 32 41 47 52  9 42 41 32  0 52 32 28 45 46  0 36 41  0 47 35


In [ ]:
def setup_dataset(dataset, author):
    print(f"Setting up dataset for {author}...")
    dataset = (
        dataset
        .shuffle(BUFFER_SIZE)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.experimental.AUTOTUNE))
    print(f"✅ Dataset ready for {author}")
    return dataset

In [ ]:
# setup_dataset() is now called inside the main training loop for each author
# vocab_ds = setup_dataset(vocab_ds)

In [ ]:
class TextGenerationModel(tf.keras.Model):

    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()

        # 1. Embedding layer: converts character indices to vectors
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        # 2. Two LSTM layers for sequence memory
        # Start with 2 layers, add more if model struggles to learn patterns
        self.lstm1 = tf.keras.layers.LSTM(rnn_units, return_sequences=True, return_state=True)
        self.lstm2 = tf.keras.layers.LSTM(rnn_units, return_sequences=True, return_state=True)

        # 3. Output layer: predicts probability of each character in vocabulary
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = self.embedding(inputs, training=training)

        # LSTM 1
        if states is None:
            states1 = None
            states2 = None
        else:
            states1 = states[:2]
            states2 = states[2:]

        x, h1, c1 = self.lstm1(x, initial_state=states1, training=training)
        x, h2, c2 = self.lstm2(x, initial_state=states2, training=training)

        # Output layer
        x = self.dense(x, training=training)

        if return_state:
            return x, [h1, c1, h2, c2]
        else:
            return x

In [ ]:
# Dataset creation is now handled inside the main training loop for each author
# if restart:
#   dataset = vocab_ds
#   del vocab_text
#   del vocab_ds
# else:
#   new_text = getRandomText(numbooks = 10)  # removed, see cell 11
#   dataset = text_to_dataset(new_text)
#   del new_text
#   dataset = setup_dataset(dataset)

In [ ]:
def build_model(vocab_size):
    model = TextGenerationModel(vocab_size, embedding_dim, rnn_units)
    print(f"✅ Model created with:")
    print(f"   vocab_size={vocab_size}")
    print(f"   embedding_dim={embedding_dim}")
    print(f"   rnn_units={rnn_units}")
    return model

In [ ]:
def verify_model(model, dataset, author, vocab_size):
    print(f"\nVerifying model output shape for {author}...")
    for input_example_batch, target_example_batch in dataset.take(1):
        example_batch_predictions = model(input_example_batch)
        print(f"Input shape:  {input_example_batch.shape} # (batch_size, sequence_length)")
        print(f"Output shape: {example_batch_predictions.shape} # (batch_size, sequence_length, vocab_size)")

        # Verify output dimensions are correct
        assert example_batch_predictions.shape[-1] == vocab_size, \
            f"Expected vocab_size={vocab_size}, got {example_batch_predictions.shape[-1]}"

        print(f"✅ Model output verified for {author}")

In [ ]:
def print_model_summary(model, author):
    print(f"\nModel summary for {author}:")
    model.summary()

    # Calculate total parameters
    total_params = model.count_params()
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
class OneStep(tf.keras.Model):
    def __init__(self, model, vocabulary, char_to_idx, idx_to_char, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        self.model = model
        self.vocabulary = vocabulary
        self.char_to_idx = char_to_idx
        self.idx_to_char = idx_to_char
        self.prediction_mask = tf.zeros(len(vocabulary))

    # Removed @tf.function - incompatible with eager operations like .numpy()
    def generate_one_step(self, inputs, states=None):
        # Convert input characters to indices
        input_chars = inputs[0].numpy().decode('utf-8')
        input_ids = tf.constant([[self.char_to_idx.get(char, 0) for char in input_chars]], dtype=tf.int64)

        # Run the model and get predictions
        predicted_logits, states = self.model(inputs=input_ids, states=states,
                                              return_state=True)
        del input_ids

        # Only use the last prediction
        predicted_logits = predicted_logits[:, -1, :]

        # Apply temperature
        predicted_logits = predicted_logits / self.temperature

        # Sample from the distribution
        predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
        del predicted_logits

        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convert indices back to characters
        predicted_chars = tf.constant([self.idx_to_char.get(int(idx), '') for idx in predicted_ids.numpy()])

        return predicted_chars, states

In [ ]:
def produce_sample(model, vocabulary, char_to_idx, idx_to_char, author, temp, epoch, prompt, num_chars=1100):
    one_step_model = OneStep(model, vocabulary, char_to_idx, idx_to_char, temp)
    states = None
    next_char = tf.constant([preprocess_text(prompt)])
    result = [tf.constant([prompt])]

    for n in range(num_chars):yys
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    result = tf.strings.join(result)
    generated_text = postprocess_text(result[0].numpy().decode('utf-8'))

    # Print to console
    print(f"\nAuthor: {author} | Epoch: {epoch} | Temp: {temp}")
    print(generated_text)

    # Save to author specific file in Google Drive
    output_file = path + f"{author}/samples.txt"
    with open(output_file, 'a', encoding='utf-8') as f:
        f.write(f"Author: {author}\n")
        f.write(f"Epoch: {epoch}\n")
        f.write(f"Temperature: {temp}\n")
        f.write(generated_text)
        f.write('\n\n')

    del states
    del next_char
    del result
    del one_step_model

## IV. Train the Model

Each author is trained separately using the same architecture defined above.
For each author the training loop will:
1. Load the author's text
2. Create the dataset
3. Build a fresh model
4. Train for num_epochs_total epochs
5. Save weights after each epoch to Google Drive
6. Generate sample text after each epoch to monitor progress

We use **sparse categorical cross entropy** as our loss function because our
model outputs logits and uses integer encoding rather than one-hot encoding.

We use **Adam** as our optimizer with a learning rate that decays slightly
each epoch to help the model converge.

In [ ]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

def train_author(author):
    print(f"\n{'='*50}")
    print(f"Training {author.upper()}")
    print(f"{'='*50}")

    # 1. Load text
    text = get_text(author)
    if text is None:
        print(f"❌ Failed to load text for {author}, skipping...")
        return

    # 2. Build or load vocabulary
    if restart:
        vocabulary, char_to_idx, idx_to_char = build_char_vocabulary(text)
        save_vocabulary(vocabulary, author)
    else:
        vocabulary, char_to_idx, idx_to_char = load_vocabulary(author)

    vocab_size = len(vocabulary)
    print(f"Vocabulary size for {author}: {vocab_size} characters")

    # 3. Create dataset
    dataset = text_to_dataset(text, author, char_to_idx)
    del text
    gc.collect()
    dataset = setup_dataset(dataset, author)

    # 4. Build fresh model with author specific vocab_size
    model = build_model(vocab_size)
    verify_model(model, dataset, author, vocab_size)
    print_model_summary(model, author)

    # 5. Load weights if continuing training
    if not restart:
        weights_path = path + f"{author}/weights_latest.weights.h5"
        if os.path.exists(weights_path):
            model.load_weights(weights_path)
            print(f"✅ Loaded weights from {weights_path}")
        else:
            print(f"⚠️ No weights found at {weights_path}, starting from scratch")

    # 6. Compile model
    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss=loss)

    # 7. Train
    start_epoch = epoch_to_pickup if (not restart and author == author_to_pickup) else 0
    for e in range(start_epoch, num_epochs_total):
        success = False
        while not success:
            try:
                print(f"\nEpoch {e+1}/{num_epochs_total}")

                # Decay learning rate
                model.optimizer.learning_rate.assign(learning_rate * (learning_rate_decay ** e))

                # Train for one epoch
                model.fit(dataset, epochs=1, verbose=1)

                # Save weights after each epoch
                epoch_weights_path = path + f"{author}/weights_epoch{e}.weights.h5"
                latest_weights_path = path + f"{author}/weights_latest.weights.h5"
                model.save_weights(epoch_weights_path)
                model.save_weights(latest_weights_path)
                print(f"✅ Weights saved for {author} epoch {e}")

                # Generate samples
                for temp in temperatures:
                    produce_sample(model, vocabulary, char_to_idx, idx_to_char, author, temp, e, sample_prompt)
                print(f"✅ Samples generated for {author} epoch {e}")

                gc.collect()
                success = True

            except Exception as ex:
                print(f"❌ Error in epoch {e}: {ex}")
                import traceback; traceback.print_exc()
                gc.collect()
                print(f"Retrying epoch {e}...")

    # 8. Cleanup
    del model
    del dataset
    del vocabulary
    del char_to_idx
    del idx_to_char
    gc.collect()
    tf.keras.backend.clear_session()
    print(f"\n✅ Training complete for {author}")

In [ ]:
# Celda 41 - Main training loop
if restart:
    # Train all authors from scratch
    for author in authors:
        train_author(author)
else:
    # Continue training from checkpoint
    # Start from author_to_pickup and continue with remaining authors
    start_index = authors.index(author_to_pickup)
    for author in authors[start_index:]:
        train_author(author)

print("\n" + "="*50)
print("ALL AUTHORS TRAINED SUCCESSFULLY ✅")
print("="*50)


Training AUSTEN
File 'austen_0.txt' found locally. Using it.
✅ austen_0.txt: 4906787 characters added
✅ Total text for austen: 4906787 characters
Vocabulary size: 58 characters
Characters:  !"&'()*,-.0123456789:;?[]^_abcdefghijklmnopqrstuvwxyz£é‘’
✅ Vocabulary saved for austen at /content/drive/My Drive/cse450_publishing/austen/vocabulary.txt
Vocabulary size for austen: 58 characters
Converting austen text to dataset...
✅ Dataset ready for austen
Setting up dataset for austen...
✅ Dataset ready for austen
✅ Model created with:
   vocab_size=58
   embedding_dim=256
   rnn_units=1024

Verifying model output shape for austen...
Input shape:  (64, 300) # (batch_size, sequence_length)
Output shape: (64, 300, 58) # (batch_size, sequence_length, vocab_size)
✅ Model output verified for austen

Model summary for austen:


Model: "text_generation_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 300, 256)         │        14,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ((64, 300, 1024), (64, │     5,246,976 │
│                                 │ 1024), (64, 1024))     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ((64, 300, 1024), (64, │     8,392,704 │
│                                 │ 1024), (64, 1024))     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 300, 58)          │        59,450 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,713,978 (52.31 MB)

 Trainable params: 13,713,978 (52.31 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 300, 256)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ((64, 300, 1024), (64, │     5,246,976 │
│                                 │ 1024), (64, 1024))     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ((64, 300, 1024), (64, │     8,392,704 │
│                                 │ 1024), (64, 1024))     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 300, 66)          │        67,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,724,226 (52.35 MB)

 Trainable params: 13,724,226 (52.35 MB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 13,724,226
Trainable parameters: 13,724,226

Epoch 1/10
87/87 ━━━━━━━━━━━━━━━━━━━━ 48s 499ms/step - loss: 2.8838
✅ Weights saved for twain epoch 0

Author: twain | Epoch: 0 | Temp: 0.4
The world seemed like such a peaceful place until the magic tree was discovered in London. The he tim hilg and I the the on the the we lo the salt ang on he mon te hand I tho ind to the and on the the the the wors and ta wome the to he we sore the his the the af the the and and If the sole the to be she thas the angere sere me the the the sood and Ton the the the he on tar he mor tad the ind I shan the to the on on me fant and an ware han the the and to the the ther the and was the the to soul wis the tor the for mos the the and ind ho ho than gas the the to the he fat the he the Ther the the the sad and the the the the in he solr the and and The win in the to the the ard the he bhe hat the bime sin dhe the the the the he whe ghan the ind in the were the wall to won the the the id the 